In [1]:
import numpy as np
import pandas as pd
import re
from sklearn.metrics.pairwise import cosine_similarity

ING_META_PATH   = "ingredient_meta.csv"
ING_EMB_PATH    = "ingredient_embeddings.npy"
BRAND_TONE_PKL  = "brand_analysis_result.pkl"

OUT_CSV  = "persona_vectors.csv"
OUT_NPY  = "persona_vectors.npy"
OUT_META = "persona_meta.csv"

CICA_WHITELIST_TOKENS = [
    "병풀","센텔라","centella","asiatica",
    "마데카","madeca","madecass",
    "asiatic","asiaticoside",
    "madecassoside","madecassic",
    "cica"
]

성분 임베딩 로드 + 정규화

In [2]:
ingredient_meta = pd.read_csv(ING_META_PATH)
ingredient_embeddings = np.load(ING_EMB_PATH)

assert "ingredient_name" in ingredient_meta.columns
assert len(ingredient_meta) == ingredient_embeddings.shape[0]

EMBEDDING_DIM = ingredient_embeddings.shape[1]
print("성분 수:", len(ingredient_meta))
print("임베딩 차원:", EMBEDDING_DIM)

def norm_ing(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[\[\]\(\)\{\}]", " ", s)
    s = re.sub(r"[^0-9a-z가-힣\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

ingredient_meta["ingredient_name_norm"] = ingredient_meta["ingredient_name"].map(norm_ing)

ingredient_embedding_dict = {}
for i, row in ingredient_meta.iterrows():
    k = row["ingredient_name_norm"]
    if k and k not in ingredient_embedding_dict:
        ingredient_embedding_dict[k] = ingredient_embeddings[i]

print("정규화 성분 dict:", len(ingredient_embedding_dict))

성분 수: 3149
임베딩 차원: 768
정규화 성분 dict: 3048


성분 기반 페르소나 벡터 생성

In [3]:
def _passes_cica_filter(k: str) -> bool:
    return any(tok in k for tok in CICA_WHITELIST_TOKENS)

def find_group_embeddings(keyword: str):
    kw = norm_ing(keyword)
    vecs, keys = [], []
    for k, v in ingredient_embedding_dict.items():
        if kw in k:
            if kw in ["시카","cica"] and not _passes_cica_filter(k):
                continue
            vecs.append(v)
            keys.append(k)
    return vecs, keys

def build_group_vector(keywords):
    all_vecs = []
    log = {}
    for kw in keywords:
        vecs, keys = find_group_embeddings(kw)
        log[kw] = keys
        all_vecs.extend(vecs)
    if all_vecs:
        return np.mean(np.stack(all_vecs), axis=0), log
    return np.zeros(EMBEDDING_DIM, dtype=np.float32), log

persona_ingredient_preference = {
    "persona_1": ["히알루론산", "시카"],
    "persona_2": ["나이아신아마이드"],
    "persona_3": ["병풀", "인삼"]
}

persona_ingredient_vector = {}
persona_match_log = {}

for pid, kws in persona_ingredient_preference.items():
    vec, log = build_group_vector(kws)
    persona_ingredient_vector[pid] = vec.astype(np.float32)
    persona_match_log[pid] = log

for pid, v in persona_ingredient_vector.items():
    print(pid, "ingredient norm:", np.linalg.norm(v))

persona_1 ingredient norm: 22.777353
persona_2 ingredient norm: 19.073088
persona_3 ingredient norm: 20.689104


브랜드톤 임베딩 로드 & 페르소나 톤 벡터 구성

In [4]:
brand_df = pd.read_pickle(BRAND_TONE_PKL)

tone_col = None
for c in brand_df.columns:
    if isinstance(brand_df[c].iloc[0], (list, np.ndarray)):
        tone_col = c
        break

if tone_col is None:
    raise ValueError("브랜드톤 임베딩 컬럼을 찾지 못함")

BRAND_COL = "브랜드" if "브랜드" in brand_df.columns else "brand"

brand_tone_dict = {
    row[BRAND_COL]: np.array(row[tone_col], dtype=np.float32)
    for _, row in brand_df.iterrows()
}

print("브랜드톤 개수:", len(brand_tone_dict))

persona_brand_map = {
    "persona_1": ["라네즈", "이니스프리"],
    "persona_2": ["설화수"],
    "persona_3": ["헤라"]
}

persona_tone_vector = {}

for pid, brands in persona_brand_map.items():
    vecs = [brand_tone_dict[b] for b in brands if b in brand_tone_dict]
    if vecs:
        persona_tone_vector[pid] = np.mean(np.stack(vecs), axis=0)
    else:
        persona_tone_vector[pid] = np.zeros(EMBEDDING_DIM, dtype=np.float32)

for pid, v in persona_tone_vector.items():
    print(pid, "tone norm:", np.linalg.norm(v))

브랜드톤 개수: 30
persona_1 tone norm: 9.447868
persona_2 tone norm: 11.596548
persona_3 tone norm: 10.985014


최종 페르소나 벡터 concat & 저장

In [5]:
persona_risk_price_vector = {
    "persona_1": [1.0, 0.9, 0.8, 0.3],
    "persona_2": [0.3, 0.2, 0.2, 0.9],
    "persona_3": [0.4, 0.3, 0.1, 0.2],
}

persona_ids = sorted(
    set(persona_tone_vector)
    & set(persona_ingredient_vector)
    & set(persona_risk_price_vector)
)

persona_final_vector = {}
rows = []

for pid in persona_ids:
    final_vec = np.concatenate([
        persona_tone_vector[pid],
        persona_ingredient_vector[pid],
        np.array(persona_risk_price_vector[pid], dtype=np.float32)
    ], axis=0)

    persona_final_vector[pid] = final_vec.astype(np.float32)

    rows.append({
        "persona_id": pid,
        "final_dim": len(final_vec)
    })

# 저장
persona_matrix = np.stack([persona_final_vector[p] for p in persona_ids])
np.save(OUT_NPY, persona_matrix)
pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
pd.DataFrame({"persona_id": persona_ids}).to_csv(OUT_META, index=False)

print("저장 완료")
print("final shape:", persona_matrix.shape)

print("\n페르소나 cosine similarity")
print(cosine_similarity(persona_matrix))

저장 완료
final shape: (3, 1540)

페르소나 cosine similarity
[[1.         0.74493265 0.9275272 ]
 [0.74493265 1.0000001  0.7422649 ]
 [0.9275272  0.7422649  0.99999976]]
